# Prepare Calendar Table

Generates a standard date-dimension table `adb_caresync.silver.calendar` spanning **1940-01-01 to 2030-12-31** with all key calendar attributes.

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# 1. Generate one row per day: 1940-01-01  to  2030-12-31
# ---------------------------------------------------------------------------
start_date = "1940-01-01"
end_date   = "2030-12-31"

date_df = (
    spark.sql(
        f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date"
    )
)

# ---------------------------------------------------------------------------
# 2. Build all standard calendar columns
# ---------------------------------------------------------------------------
calendar_df = date_df.select(
    # ---- Key & base date ----
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),

    # ---- Year ----
    F.year("date").alias("year"),
    F.trunc("date", "year").alias("first_day_of_year"),
    F.to_date(F.concat(F.year("date"), F.lit("-12-31"))).alias("last_day_of_year"),

    # ---- Quarter ----
    F.quarter("date").alias("quarter"),
    F.concat(F.lit("Q"), F.quarter("date")).alias("quarter_name"),
    F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")).alias("year_quarter"),
    F.trunc("date", "quarter").alias("first_day_of_quarter"),
    F.date_add(F.add_months(F.trunc("date", "quarter"), 3), -1).alias("last_day_of_quarter"),

    # ---- Month ----
    F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_name_short"),
    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.trunc("date", "month").alias("first_day_of_month"),
    F.last_day("date").alias("last_day_of_month"),
    (F.datediff(F.last_day("date"), F.trunc("date", "month")) + 1).alias("days_in_month"),

    # ---- Week ----
    F.weekofyear("date").alias("week_of_year"),
    F.date_trunc("week", F.col("date")).cast("date").alias("first_day_of_week"),   # Monday
    F.date_add(F.date_trunc("week", F.col("date")), 6).cast("date").alias("last_day_of_week"),

    # ---- Day ----
    F.dayofmonth("date").alias("day_of_month"),
    F.dayofweek("date").alias("day_of_week"),          # 1=Sun … 7=Sat
    ((F.dayofweek("date") + 5) % 7 + 1).alias("iso_day_of_week"),  # 1=Mon … 7=Sun
    F.dayofyear("date").alias("day_of_year"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_name_short"),

    # ---- Weekend / Weekday ----
    F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("date").isin(1, 7), "Weekend").otherwise("Weekday").alias("day_type"),

    # ---- Fiscal year (April start) ----
    F.when(F.month("date") >= 4, F.year("date") + 1)
     .otherwise(F.year("date")).alias("fiscal_year"),
    F.when(F.month("date") >= 4, F.ceil((F.month("date") - 3) / 3))
     .otherwise(F.ceil((F.month("date") + 9) / 3))
     .cast("int").alias("fiscal_quarter"),

    # ---- Leap year ----
    F.when(
        (F.year("date") % 4 == 0) &
        ((F.year("date") % 100 != 0) | (F.year("date") % 400 == 0)),
        True
    ).otherwise(False).alias("is_leap_year"),
)

# ---------------------------------------------------------------------------
# 3. Write to Unity Catalog
# ---------------------------------------------------------------------------
calendar_df.write.format("delta").mode("overwrite").saveAsTable("adb_caresync.silver.calendar")

print(f"✔ adb_caresync.silver.calendar written — {calendar_df.count():,} rows")
display(spark.table("adb_caresync.silver.calendar").limit(10))